In [ ]:
%%capture
!pip install pydantic-ai -q
from openai import AsyncAzureOpenAI
from google.colab import userdata
client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))
from pydantic import BaseModel
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.models.openai import OpenAIModel
import nest_asyncio
nest_asyncio.apply()
model = OpenAIModel('gpt-4o', openai_client=client)

In [ ]:
import json
import os
from dataclasses import asdict, dataclass

import yaml


@dataclass
class Task:
    title: str
    isDone: bool = False


def read_tasks(userid: str) -> str:
    """
    Reads all tasks from a JSON file and returns the result as a YAML string.

    Args:
        userid (str): The user ID whose tasks should be read.

    Returns:
        str: A YAML-formatted string of the tasks.
    """
    file_path = f"tasks/{userid}.json"
    if not os.path.exists(file_path):
        return yaml.dump([])

    with open(file_path, "r") as file:
        tasks = json.load(file)
    return yaml.dump(tasks, sort_keys=False)


def mark_task_as_done(userid: str, title: str) -> str:
    """
    Marks a task as done by title and updates the JSON file.

    Args:
        userid (str): The user ID whose tasks should be updated.
        title (str): The title of the task to mark as done.

    Returns:
        str: A message indicating the task was successfully updated.

    Raises:
        FileNotFoundError: If the user's task file does not exist.
        ValueError: If no task with the given title is found.
    """
    file_path = f"tasks/{userid}.json"
    if not os.path.exists(file_path):
        raise FileNotFoundError("Task file not found.")

    with open(file_path, "r") as file:
        tasks = json.load(file)

    task_found = False
    for task in tasks:
        if task["title"] == title:
            task["isDone"] = True
            task_found = True
            break

    if not task_found:
        raise ValueError("Task with the given title not found.")

    with open(file_path, "w") as file:
        json.dump(tasks, file, indent=2)

    return "Successfully updated task"


def add_task(userid: str, title: str) -> str:
    """
    Appends a new task to the task list and updates the JSON file.

    Args:
        userid (str): The user ID whose task list should be updated.
        title (str): The title of the new task to add.

    Returns:
        str: A message indicating the task was successfully added.

    Raises:
        ValueError: If a task with the same title already exists.
    """
    file_path = f"tasks/{userid}.json"
    tasks = []

    if os.path.exists(file_path):
        with open(file_path, "r") as file:
            tasks = json.load(file)

    for task in tasks:
        if task["title"] == title:
            raise ValueError("Task with the same title already exists.")

    new_task = Task(title=title)
    tasks.append(asdict(new_task))

    with open(file_path, "w") as file:
        json.dump(tasks, file, indent=2)
    print('success')
    return "Successfully added task"

In [ ]:
from dataclasses import dataclass


@dataclass
class UserState:
    id: str

In [ ]:
import os

tool_add_task = add_task
tool_mark_task_as_done = mark_task_as_done
tool_read_tasks = read_tasks

task_executor_agent = Agent(
    model,
    system_prompt=(
        "You are a helpful ai assistant\n"
        "Follow the entire conversation history carefully to identify the tool to call and arguments to pass to them."
        "Do not call the function if the action is already performed"
    ),
    deps_type=UserState,
)


@task_executor_agent.tool
def read_tasks(ctx: RunContext[UserState]) -> str:
    """
    Reads all tasks
    """

    if not isinstance(ctx, str) and isinstance(ctx.deps, UserState):
        return f"Here's all the requested tasks:\n{tool_read_tasks(ctx.deps.id)}"


@task_executor_agent.tool
def add_task(ctx: RunContext[UserState], title: str) -> str:
    """
    Appends a new task to the task list

    Args:
        title (str): The title of the new task to add.
    """
    if not isinstance(ctx, str) and isinstance(ctx.deps, UserState):
        return tool_add_task(ctx.deps.id, title)


@task_executor_agent.tool
def tool_mark_task_as_done(ctx: RunContext[UserState], title: str) -> str:
    """
    Marks a task as done in the task list

    Args:
        title (str): The title of the task to mark as done.
    """
    if not isinstance(ctx, str) and isinstance(ctx.deps, UserState):
        return tool_mark_task_as_done(ctx.deps.id, title)

In [ ]:
user_state = UserState(id="YourTechBud")
task_executor_agent.run_sync("Get the tasks", deps=user_state).data

'Here are the current tasks:\n\n1. Buy groceries - Not Done\n2. Finish the project report - Done\n3. Walk the dog - Not Done\n4. Prepare dinner - Done\n5. Call the plumber - Not Done'

In [ ]:
import os

from pydantic import BaseModel
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.models.openai import OpenAIModel

# Define a Pydantic model for the intent
class Intent(BaseModel):
    action: str

# Create a PydanticAI instance
intent_classifier_agent = Agent(
    model,
    system_prompt=(
        "You are a helpful ai assistant\n"
        "Identify the user's intent from the provided message\n"
        "Choose from these options: `getTasks`, `markTaskAsDone`, `addTask` and `unknown`\n"
        "If the user mentions that they have done something, mark the task as done."
        "Action will be invalid for all invalid messages"
    ),
    result_type=Intent,
    result_tool_name="user_intent",
    result_tool_description="Log the user's intended action and whether it is valid",
    result_retries=3,
)


@intent_classifier_agent.result_validator
def validate_result(ctx: RunContext[None], result: Intent) -> Intent:
    if result.action not in ["getTasks", "markTaskAsDone", "addTask", "unknown"]:
        raise ModelRetry(
            "Invalid action. Please choose from `getTasks`, `markTaskAsDone`, `addTask` and `unknown`"
        )

    return result

In [ ]:
import os

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel

task_summarizer_agent = Agent(
    model,
    system_prompt="You are a helpful AI assitant and a productivity expert. You always respond with motivational and helpful messages.",
)

In [187]:
import os

from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIModel
from pydantic import  BaseModel, Field

# Define a Pydantic model for the result
class Match(BaseModel):
    title: str = Field(description="The most similar title of the task or empty")
    is_title_present: bool = Field(description="Whether the most similar title is present in the list of titles")


# Create a PydanticAI instance
title_matcher_agent = Agent(
    model,
    system_prompt=("You are a helpful ai assistant\n"),
    deps_type=UserState,
    result_type=Match,
    result_tool_name="title_matcher",
    result_tool_description="Identify the right title",
    result_retries=3,
)


@title_matcher_agent.system_prompt
def system_prompt(ctx: RunContext[UserState]) -> str:
    # First, we read the tasks for the user
    titles = tool_read_tasks(ctx.deps.id)
    print(ctx.deps.id, titles)
    # Craft the system prompt
    return (
        "Identify the title provided by the user\n"
        f"The title must be present in this list: <titles>{titles}</titles>\n\n"
        "Return a title from the previous which is the closest or related to the user's input\n"
        "If no title is present, provide an empty string and mark is_title_present as False"
        "Remember the user could use similar expressions that means a title"
    )

In [188]:
!mkdir tasks

mkdir: cannot create directory ‘tasks’: File exists


In [189]:
%%writefile tasks/YourTechBud.json
[
  {
    "title": "Buy groceries",
    "isDone": false
  },
  {
    "title": "Finish the project report",
    "isDone": true
  },
  {
    "title": "Walk the dog",
    "isDone": false
  },
  {
    "title": "Prepare dinner",
    "isDone": true
  },
  {
    "title": "Call the plumber",
    "isDone": false
  }
]

Overwriting tasks/YourTechBud.json


In [193]:
from time import sleep

# Wait for logfire to start
sleep(1)

# Create a UserState instance
user_state = UserState(id="YourTechBud")

while query := input("Enter your query: "):
    if query == "exit":
        break

    # Identify the intent
    result = intent_classifier_agent.run_sync(query)
    print(f"Identified intent: {result.data.action}")

    match result.data.action:
        case "addTask":
            result = task_executor_agent.run_sync(
                "Identify the right title and add the task", deps=user_state
            )

        case "getTasks":
            result = task_executor_agent.run_sync("Get the tasks", deps=user_state)
            result = task_summarizer_agent.run_sync(
                f"Summarize these tasks in a concise and oraganized manner: {result.data}"
            )

        case "markTaskAsDone":
            result = title_matcher_agent.run_sync(query, deps=user_state)
            print(
                f"Title: {result.data.title}, Is title present: {result.data.is_title_present}"
            )
            if not result.data.is_title_present:
                raise Exception("Title not present")

            result = task_executor_agent.run_sync(
                f"Mark the task '{result.data.title}' as done",
                deps=user_state,
            )

    print(result.data)
# Get the tasks
# I walked the hell outta that dog
# I called the plumber

Enter your query: Get the tasks
Identified intent: getTasks
Certainly! Here's a concise and organized summary of your current tasks:

**Incomplete Tasks:**
1. Buy groceries
2. Walk the dog
3. Call the plumber

**Completed Tasks:**
1. Finish the project report
2. Prepare dinner

Keep up the great work on completing some of your tasks! Stay focused and you'll get through the rest in no time. 🎉💪
Enter your query: I walked the hell outta that dog
Identified intent: markTaskAsDone
YourTechBud - title: Buy groceries
  isDone: false
- title: Finish the project report
  isDone: true
- title: Walk the dog
  isDone: false
- title: Prepare dinner
  isDone: true
- title: Call the plumber
  isDone: false

Title: I walked the hell outta that dog, Is title present: True
The task "I walked the hell outta that dog" has been marked as done.
Enter your query: exit


In [194]:
from pprint import pprint
pprint(result.all_messages())

[ModelRequest(parts=[SystemPromptPart(content='You are a helpful ai assistant\n'
                                              'Follow the entire conversation '
                                              'history carefully to identify '
                                              'the tool to call and arguments '
                                              'to pass to them.Do not call the '
                                              'function if the action is '
                                              'already performed',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content="Mark the task 'I walked the hell "
                                            "outta that dog' as done",
                                    timestamp=datetime.datetime(2025, 2, 11, 15, 8, 11, 295590, tzinfo=datetime.timezone.utc),
                                    part_kind='user-promp